# ResNet — Deep Residual Learning for Image Recognition (2015)

_Papers · Computer Vision_

**Add a layer's input back to its output, so networks hundreds of layers deep finally train.**

---

This notebook is a practice scaffold for the **ResNet — Deep Residual Learning for Image Recognition (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as T

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example: y = F(x) + x, then ReLU. ---
x   = torch.tensor([0.5, -0.3])
Fx  = torch.tensor([-0.2, 0.1])          # what the two layers compute (the residual)
y   = Fx + x                              # Eqn. 1: add the shortcut
out = torch.relu(y)
print("worked example:  sum =", y.tolist(), " ReLU =", out.tolist())
# worked example:  sum = [0.30000001192092896, -0.19999998807907104]  ReLU = [0.30000001192092896, 0.0]


# --- 1. The residual block (built by hand). skip=True -> ResNet; skip=False -> plain ablation. ---
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, skip=True):
        super().__init__()
        self.skip  = skip
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        # Projection shortcut (Eqn. 2, W_s): only when shape changes.
        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))   # conv -> BN -> ReLU
        out = self.bn2(self.conv2(out))            # conv -> BN  (F(x))
        if self.skip:                              # the ablation switch
            if self.proj is not None:
                identity = self.proj(x)            # W_s x  when dims differ
            out = out + identity                   # Eqn. 1: F(x) + x
        return self.relu(out)                      # second nonlinearity AFTER the add


# --- 2. Stack blocks into stages -> a small ResNet (or plain net if skip=False). ---
class SmallResNet(nn.Module):
    def __init__(self, blocks_per_stage=3, skip=True, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        def stage(in_ch, out_ch, stride):
            layers = [BasicBlock(in_ch, out_ch, stride, skip)]
            layers += [BasicBlock(out_ch, out_ch, 1, skip) for _ in range(blocks_per_stage - 1)]
            return nn.Sequential(*layers)
        self.stage1 = stage(16, 16, 1)
        self.stage2 = stage(16, 32, 2)             # downsample + double channels -> projection
        self.stage3 = stage(32, 64, 2)
        self.head   = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage3(self.stage2(self.stage1(x)))
        x = x.mean(dim=(2, 3))                      # global average pool
        return self.head(x)


# --- 3. A CIFAR-10 subset (torchvision is preinstalled in Colab). ---
tfm = T.Compose([T.ToTensor(),
                 T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=tfm)
subset = torch.utils.data.Subset(full, range(4000))     # small + fast
loader = torch.utils.data.DataLoader(subset, batch_size=128, shuffle=True)
device = "cuda" if torch.cuda.is_available() else "cpu"


def train(skip, epochs=4, depth=3):
    torch.manual_seed(0)
    net = SmallResNet(blocks_per_stage=depth, skip=skip).to(device)
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    lf  = nn.CrossEntropyLoss()
    net.train(); curve = []
    for ep in range(epochs):
        tot = 0.0; nb = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = lf(net(xb), yb); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        curve.append(tot / nb)
        print(f"  epoch {ep}  train loss {curve[-1]:.4f}")
    return curve

print("RESIDUAL net (skip=True):")
res_curve = train(skip=True)
print("PLAIN net (skip=False, ABLATION  -- same depth, no '+ identity'):")
plain_curve = train(skip=False)

print("\nResidual train loss per epoch:", [round(c, 4) for c in res_curve])
print("Plain    train loss per epoch:", [round(c, 4) for c in plain_curve])
# The residual curve falls faster and lower; the matched plain net lags -- the degradation effect.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_At depth, does the training loss fall for a residual net vs a matched plain net (same depth, no skip)?_

In [ ]:
import torch, torch.nn as nn, numpy as np

# Two matched deep nets, identical except for the residual skip. Reproduces the
# qualitative effect on toy data (degradation when the skip is removed).
N, K, n, D = 256, 3, 32, 50
g = torch.Generator().manual_seed(1)
y = torch.randint(0, K, (N,), generator=g)
centers = torch.randn(K, n, generator=g) * 2.0
X = centers[y] + torch.randn(N, n, generator=g) * 0.5

class Block(nn.Module):
    def __init__(self, n, residual):
        super().__init__()
        self.fc = nn.Linear(n, n, bias=False)
        self.bn = nn.BatchNorm1d(n)
        self.residual = residual
    def forward(self, x):
        out = self.bn(self.fc(x))
        return torch.relu(out + x if self.residual else out)   # the only difference

class Net(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.stem   = nn.Linear(n, n)
        self.blocks = nn.Sequential(*[Block(n, residual) for _ in range(D)])
        self.head   = nn.Linear(n, K)
    def forward(self, x):
        return self.head(self.blocks(torch.relu(self.stem(x))))

def train(residual, steps=120, lr=0.03):
    torch.manual_seed(0)
    net = Net(residual); net.train()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lf  = nn.CrossEntropyLoss(); losses = []
    for t in range(steps):
        opt.zero_grad(); loss = lf(net(X), y); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

plain = train(residual=False)
resid = train(residual=True)
idx = np.linspace(0, 119, 30).astype(int)
print("Plain   :", [[int(i), round(plain[i], 4)] for i in idx])
print("Residual:", [[int(i), round(resid[i], 4)] for i in idx])
# Residual -> ~0 within ~5 steps and stays there.
# Plain    -> crawls, plateaus, spikes back to ~1.17 around step 106 (degradation).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working tiny ResNet whose deep version trains well. Remove the
            single line + identity from the block (turning it into a matched plain net of the
            same depth) and retrain. What happens to the training curve, and what does that
            demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only the skip: change out = relu(out + identity) to out = relu(out); keep depth, width, optimizer, and data identical. — _An honest ablation changes exactly one thing &mdash; the residual connection &mdash; so any difference is attributable to it._
- Retrain and watch the deep plain net's training loss: it falls slowly, plateaus high, or oscillates, while the residual version dropped fast and stayed low. — _Without the "+1" highway the gradient to early layers is weak, so the deep plain stack cannot optimize &mdash; the degradation problem reproduced on toy data._
- Conclude that the skip, not extra parameters, is what made the deep net trainable. — _Both nets have the same layers and parameter count; only the residual one optimizes, isolating the skip as the cause._

**Answer:** The deep plain net's training loss lags badly &mdash; it stays high and noisy while
                 the residual net converges. Since the two nets are identical except for the "+ identity",
                 this isolates the skip connection as the reason deep nets train: it is an
                 optimization fix (the degradation problem), not a parameter-count or regularization
                 effect. The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** A block takes a 16-channel, 8&times;8 feature map and must output a 32-channel, 4&times;4 map
            (it downsamples and doubles channels). You write out = relu(out + x) and it errors.
            Why, and how do you fix the shortcut?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check shapes: out is (32, 4, 4) but x is (16, 8, 8); element-wise add needs equal shapes. — _The shortcut and main path must match in channels and spatial size to be summed._
- Build a projection shortcut: a $1\times1$ nn.Conv2d(16, 32, kernel_size=1, stride=2) (+ BatchNorm), and set identity = self.proj(x). — _Eqn. 2's $W_s$: the $1\times1$ conv with stride 2 maps 16&rarr;32 channels and halves the spatial size, matching out._
- Now add the projected identity and apply ReLU. — _Shapes agree, so $\sigma(F(x) + W_s x)$ is well-defined._

**Answer:** The identity wire cannot be added because the shapes differ (16&times;8&times;8 vs
                 32&times;4&times;4). Replace it with a projection shortcut (Eqn. 2): a $1\times1$
                 convolution with stride 2 mapping 16&rarr;32 channels, so $W_s x$ matches the block's output
                 and the residual sum works.

</details>

**Problem 3.** Your worked example had $x = [0.5, -0.3]$ and residual $F(x) = [-0.2, 0.1]$, giving block output
            $[0.3, 0.0]$. Suppose instead the layers learned $F(x) = [0, 0]$. What does the block output, and
            why is that the "safe fallback" the paper relies on?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into Eqn. 1: $y = F(x) + x = [0,0] + [0.5,-0.3] = [0.5,-0.3]$. — _With a zero residual the block reduces to the identity plus the final activation._
- Apply the final ReLU: $\mathrm{ReLU}([0.5,-0.3]) = [0.5, 0.0]$. — _The positive component passes; the negative is clipped &mdash; but the input's signal $0.5$ survives._
- Note that driving $F$ to zero is trivial for the layers (just push the weights small), so the network can always recover a near-identity block. — _That is why adding depth can never make optimization strictly worse: the extra blocks can opt out by learning $F=0$._

**Answer:** With $F(x)=[0,0]$ the block outputs $\mathrm{ReLU}([0.5,-0.3]) = [0.5, 0.0]$ &mdash; it just
                 passes the input through. Learning $F=0$ is easy (small weights), so a residual block can
                 cheaply become a near-identity. That free "do nothing safely" option is exactly what a deep
                 plain stack struggles to learn, and why residual depth is safe.

</details>